In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

if os.path.exists("data"):
    DATA_DIR = "data"
elif os.path.exists("../data"):
    DATA_DIR = "../data"

# Папка для моделей
MODELS_DIR = os.path.join(os.path.dirname(DATA_DIR), "models") if DATA_DIR == "../data" else "models"
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"✅ Папка данных: {DATA_DIR}")
print(f"✅ Папка моделей: {MODELS_DIR}")

✅ Папка данных: ../data
✅ Папка моделей: ..\models


In [2]:
# Загружаем синтетику
df = pd.read_csv(f"{DATA_DIR}/synthetic_data.csv")

feature_cols = ['income', 'expense', 'debt_ratio', 'balance', 
                'active_cards', 'days_on_platform', 'closed_credits', 'has_overdue']

X = df[feature_cols]
y = df['default']

# Масштабирование
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Обучаем 2 модели
logreg = LogisticRegression(random_state=42, max_iter=1000)
logreg.fit(X_scaled, y)

rf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)
rf.fit(X, y)  # RF не нужен scaler

print("✅ Модели обучены:")
print(f"   Logistic Regression — коэффициенты: {len(logreg.coef_[0])}")
print(f"   Random Forest — деревьев: {len(rf.estimators_)}")

✅ Модели обучены:
   Logistic Regression — коэффициенты: 8
   Random Forest — деревьев: 100


In [3]:
# Сохраняем LogReg (лучшая по ROC-AUC в предыдущем запуске)
joblib.dump(logreg, f"{MODELS_DIR}/logreg_model.pkl")
joblib.dump(scaler, f"{MODELS_DIR}/scaler.pkl")
joblib.dump(rf, f"{MODELS_DIR}/rf_model.pkl")

# Сохраняем список признаков (важно для использования модели!)
with open(f"{MODELS_DIR}/feature_cols.txt", "w") as f:
    f.write("\n".join(feature_cols))

print(f"✅ Модели сохранены в {MODELS_DIR}/")
print(f"   - logreg_model.pkl")
print(f"   - scaler.pkl")
print(f"   - rf_model.pkl")
print(f"   - feature_cols.txt")

✅ Модели сохранены в ..\models/
   - logreg_model.pkl
   - scaler.pkl
   - rf_model.pkl
   - feature_cols.txt


In [5]:
# Проверяем, что модель загружается и работает
loaded_model = joblib.load(f"{MODELS_DIR}/logreg_model.pkl")
loaded_scaler = joblib.load(f"{MODELS_DIR}/scaler.pkl")

# Пробуем предсказать на одном примере
test_features = pd.DataFrame(
    [[22000, 8000, 0.36, 180000, 2, 300, 1, 0]],
    columns=['income', 'expense', 'debt_ratio', 'balance', 
             'active_cards', 'days_on_platform', 'closed_credits', 'has_overdue']
)
test_scaled = loaded_scaler.transform(test_features)
proba = loaded_model.predict_proba(test_scaled)[0, 1]
rate = round(5 + proba * 20, 2)

print(f"🧪 Тест модели:")
print(f"   Вероятность дефолта: {proba:.3f}")
print(f"   Ставка: {rate}%")
print(f"\n✅ Модель работает корректно!")

🧪 Тест модели:
   Вероятность дефолта: 0.154
   Ставка: 8.09%

✅ Модель работает корректно!
